In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
from torch.optim import AdamW, lr_scheduler
import torch.nn as nn
from model import AliLM
from utils import data_loading
from einops import rearrange

In [ ]:
token_sequence = torch.concat([torch.load(f"./data/ts_train_shard_0000{i}.pt") for i in range(3)])
# token_sequence = torch.concat(
#     [token_sequence, torch.load(f"./data/ts_train_shard_00010.pt")]
# )
print(f"Number of tokens: {len(token_sequence):,}")
vocabulary_size = 50257
print(f"Vocabulary size: {vocabulary_size:,}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
config = {
    "vocab_size": vocabulary_size,
    "context_length": 256,
    "embedding_dim": 500,
    "num_heads": 10,
    "num_layers": 30,
    "d_ff": 5,
    "Batch_size": 16,
    "learning_rate": 1e-4}

In [ ]:
model = AliLM(vocab_size=config["vocab_size"],
              context_length=config["context_length"],
              d_model=config["embedding_dim"],
              num_heads=config["num_heads"],
              num_layers=config["num_layers"],
              d_ff=config["d_ff"]).to(device).to(dtype)



In [ ]:
def param_count(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def model_size(model):
    return sum(p.numel() * p.element_size() for p in model.parameters() if p.requires_grad) / (1024 ** 2)

def save_model(model, path):
    torch.save(model.state_dict(), path)

In [ ]:
print(f"Number of trainable parameters: {param_count(model):,}")
print(f"Model size (MB): {model_size(model):.2f}")
print(f"device: {device}")
print(f"dtype: {dtype}")

In [ ]:
total_steps = 5000
loss = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=1e-4)
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

In [ ]:
total_loss = 0.0
num_batches = 0
loss_values = []
model.train()

In [ ]:


for epoch in range(total_steps):

    
    X, Y = data_loading(token_sequence, batch_size= config["Batch_size"], context_length=config["context_length"], device = device)
        
    optimizer.zero_grad()
    
    logits = model(X)
    
    loss_value = loss(logits.view(-1, config["vocab_size"]), Y.view(-1).to(torch.long))
    
    loss_values.append(loss_value.item())
    
    loss_value.backward()
    
    # Clip gradients to prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    
    optimizer.step()
    
    scheduler.step()
    total_loss += loss_value.item()
    num_batches += 1
    average_loss = total_loss / num_batches
    perplexity = torch.exp(torch.tensor(average_loss)).item()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch + 1}, Average Loss: {average_loss:.4f}, Perplexity: {perplexity:.4f}")
        

In [ ]:
import matplotlib.pyplot as plt

plt.plot(loss_values)
plt.xlabel("Batch")
plt.ylabel("Loss")
plt.title("Training Loss Over Batches")
plt.show()

In [ ]:
# generation
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")



In [ ]:
# save_model(model, "model_weights_untrained.pt")

In [ ]:
import torch


prompt = " "

inputs = tokenizer.encode(prompt, return_tensors="pt").to(device)
token_text = prompt
for i in range(50):
    
    print(token_text, end="", flush=True)
    with torch.no_grad():
        logits = model(inputs)
        probabilities = torch.nn.functional.softmax(logits[:, -1, :], dim=-1)
        predicted_token_id = torch.multinomial(probabilities, num_samples=1)

    inputs = torch.cat([inputs, predicted_token_id], dim=1)

    token_text = tokenizer.decode(predicted_token_id[0], skip_special_tokens=True)

    # heuristic newline insertion
    if i % 20 == 0:
        print("\n", end="", flush=True)
print("\n")
print(f"context length: {inputs.shape[1]}", end="\r", flush=True)

In [ ]:
torch.cuda.is_available()